# Module 3 - Session 4: Chunking Strategies & Retrieval Quality
Goal: Compare 3 chunking strategies and measure which gives the best retrieval scores.

In [1]:
# Install all libraries needed for this session
# No new libraries today — everything is from previous sessions
# sentence-transformers — convert text to vectors
# faiss-cpu — vector database
# pymupdf — read PDF
# groq — LLM for final answer
!pip install -q sentence-transformers faiss-cpu pymupdf groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.4 MB/s eta 0:00:00


## Step 2 - Load Embedding Model and Recreate Swiggy Policy PDF
Same PDF as Session 3 — we recreate it here so this notebook is self contained.
A good portfolio notebook should run completely on its own without depending on other notebooks.

In [3]:
import fitz
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")

# Recreate Swiggy policy PDF
policy_sections = [
    {
        "title": "Section 1: Delivery Partner Eligibility",
        "content": "To become a Swiggy delivery partner, you must be at least 18 years old. You must own a smartphone with Android 6.0 or above. A valid driving license and vehicle registration certificate are mandatory. Two-wheeler or bicycle ownership is required for deliveries. You must have a valid Aadhaar card and PAN card for verification. Background verification will be conducted before onboarding. Partners must complete the Swiggy training module before first delivery."
    },
    {
        "title": "Section 2: Earnings and Payments",
        "content": "Delivery partners earn a base fee for every order delivered. Additional incentives are provided for completing surge orders during peak hours. Weekly payments are credited directly to the registered bank account. Partners can track their daily and weekly earnings in the Swiggy Partner app. Surge bonuses are calculated based on orders completed in high demand zones. Tips provided by customers are credited in full to the delivery partner. Payment disputes must be raised within 7 days through the partner support portal."
    },
    {
        "title": "Section 3: Order Handling and Delivery Rules",
        "content": "Delivery partners must pick up orders within 5 minutes of reaching the restaurant. Orders must be delivered within the estimated time shown in the app. Partners must not open or tamper with food packaging at any time. If a customer is unreachable, partners must wait 5 minutes before contacting support. Proof of delivery photo must be uploaded for all orders above Rs 500. Partners must follow the route suggested by the Swiggy navigation system. Any delivery delay beyond 15 minutes must be reported through the app."
    },
    {
        "title": "Section 4: Code of Conduct",
        "content": "Delivery partners must maintain professional behavior with customers at all times. Use of abusive language or aggressive behavior will result in immediate suspension. Partners must wear the Swiggy delivery uniform and helmet during all deliveries. Mobile phones must not be used while riding the delivery vehicle. Partners must not accept orders outside the Swiggy platform. Any form of fraud including fake deliveries will result in permanent deactivation. Partners must report any accident or incident immediately through the app."
    },
    {
        "title": "Section 5: Grievance and Support",
        "content": "Delivery partners can raise grievances through the Swiggy Partner app support section. All grievances will be acknowledged within 24 hours of submission. Resolution of grievances will be provided within 5 working days. Partners can escalate unresolved issues to the regional partner support team. Swiggy provides accident insurance coverage for all active delivery partners. Medical emergencies during delivery must be reported within 48 hours. Partners have the right to appeal any suspension or deactivation decision."
    }
]

# Create PDF with proper text wrapping
doc = fitz.open()
for section in policy_sections:
    page = doc.new_page()
    page.insert_text((50, 50), section["title"], fontsize=14, color=(0, 0, 0))
    rect = fitz.Rect(50, 80, 550, 750)
    page.insert_textbox(rect, section["content"], fontsize=10, color=(0, 0, 0))

pdf_path = "/content/swiggy_delivery_policy.pdf"
doc.save(pdf_path)
doc.close()
print(f"PDF created at: {pdf_path}")

# Extract full text as one single string
doc = fitz.open(pdf_path)
full_text = ""
for page_num in range(len(doc)):
    full_text += doc[page_num].get_text()
doc.close()

print(f"Total characters extracted: {len(full_text)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!
PDF created at: /content/swiggy_delivery_policy.pdf
Total characters extracted: 2735


## Step 3 - Strategy 1: Fixed Size Chunking
Split text every N characters regardless of sentence boundaries.
Simple and fast but can cut sentences in the middle.

In [4]:
def fixed_size_chunks(text, chunk_size=200, overlap=0):
    """
    Split text into fixed size chunks of N characters.
    chunk_size — how many characters per chunk
    overlap — how many characters to repeat from previous chunk
    """
    chunks = []
    start = 0

    while start < len(text):
        # Take chunk_size characters starting from current position
        end = start + chunk_size
        chunk = text[start:end]

        # Only add if chunk has actual content
        if chunk.strip():
            chunks.append(chunk.strip())

        # Move forward by chunk_size minus overlap
        # overlap=0 means no repetition between chunks
        start = end - overlap

    return chunks

# Apply fixed size chunking to our full text
fixed_chunks = fixed_size_chunks(full_text, chunk_size=200, overlap=0)

print(f"Strategy 1 — Fixed Size Chunking")
print(f"Total chunks created: {len(fixed_chunks)}")
print(f"\n--- Preview of Chunk 1 ---")
print(fixed_chunks[0])
print(f"\n--- Preview of Chunk 2 ---")
print(fixed_chunks[1])

Strategy 1 — Fixed Size Chunking
Total chunks created: 14

--- Preview of Chunk 1 ---
Section 1: Delivery Partner Eligibility
To become a Swiggy delivery partner, you must be at least 18 years old. You must own a smartphone with
Android 6.0 or above. A valid driving license and vehicle

--- Preview of Chunk 2 ---
registration certificate are mandatory. Two-wheeler or
bicycle ownership is required for deliveries. You must have a valid Aadhaar card and PAN card for verification.
Background verification will be


## Step 4 - Strategy 2: Sentence Chunking
Split text at every sentence boundary using full stops.
Clean and meaningful — each chunk is a complete thought.

In [5]:
def sentence_chunks(text):
    """
    Split text at sentence boundaries.
    Each chunk is one complete sentence.
    We split on ". " to avoid splitting decimal numbers like "Rs 500."
    """
    # Split on period followed by space — clean sentence boundary
    sentences = text.split(". ")

    chunks = []
    for sentence in sentences:
        # Clean up whitespace and newlines
        sentence = sentence.strip().replace("\n", " ")

        # Only add if sentence has meaningful content
        # We skip very short chunks under 20 characters
        if len(sentence) > 20:
            chunks.append(sentence)

    return chunks

# Apply sentence chunking to our full text
sentence_chunk_list = sentence_chunks(full_text)

print(f"Strategy 2 — Sentence Chunking")
print(f"Total chunks created: {len(sentence_chunk_list)}")
print(f"\n--- Preview of Chunk 1 ---")
print(sentence_chunk_list[0])
print(f"\n--- Preview of Chunk 2 ---")
print(sentence_chunk_list[1])
print(f"\n--- Preview of Chunk 3 ---")
print(sentence_chunk_list[2])

Strategy 2 — Sentence Chunking
Total chunks created: 28

--- Preview of Chunk 1 ---
Section 1: Delivery Partner Eligibility To become a Swiggy delivery partner, you must be at least 18 years old

--- Preview of Chunk 2 ---
You must own a smartphone with Android 6.0 or above

--- Preview of Chunk 3 ---
A valid driving license and vehicle registration certificate are mandatory


## Step 5 - Strategy 3: Sliding Window Chunking
Group sentences together with overlap between chunks.
Each chunk has 3 sentences, next chunk starts 1 sentence later.
This captures context that would be lost at chunk boundaries.

In [6]:
def sliding_window_chunks(text, window_size=3, step=1):
    """
    Group sentences into overlapping windows.
    window_size — how many sentences per chunk
    step — how many sentences to move forward each time

    Example with window_size=3, step=1:
    Chunk 1 — sentences 1, 2, 3
    Chunk 2 — sentences 2, 3, 4
    Chunk 3 — sentences 3, 4, 5
    Sentence 2 and 3 appear in both Chunk 1 and Chunk 2
    This overlap preserves context across boundaries
    """
    # First split into sentences — reuse our sentence splitter
    sentences = text.split(". ")
    sentences = [s.strip().replace("\n", " ") for s in sentences if len(s.strip()) > 20]

    chunks = []
    # Slide the window across all sentences
    for i in range(0, len(sentences) - window_size + 1, step):
        # Take window_size sentences starting at position i
        window = sentences[i:i + window_size]

        # Join them back into one chunk
        chunk = ". ".join(window)

        if chunk.strip():
            chunks.append(chunk.strip())

    return chunks

# Apply sliding window chunking
sliding_chunks = sliding_window_chunks(full_text, window_size=3, step=1)

print(f"Strategy 3 — Sliding Window Chunking")
print(f"Total chunks created: {len(sliding_chunks)}")
print(f"\n--- Preview of Chunk 1 ---")
print(sliding_chunks[0])
print(f"\n--- Preview of Chunk 2 ---")
print(sliding_chunks[1])

Strategy 3 — Sliding Window Chunking
Total chunks created: 26

--- Preview of Chunk 1 ---
Section 1: Delivery Partner Eligibility To become a Swiggy delivery partner, you must be at least 18 years old. You must own a smartphone with Android 6.0 or above. A valid driving license and vehicle registration certificate are mandatory

--- Preview of Chunk 2 ---
You must own a smartphone with Android 6.0 or above. A valid driving license and vehicle registration certificate are mandatory. Two-wheeler or bicycle ownership is required for deliveries


## Step 6 - Compare All 3 Chunking Strategies
We test the same 4 questions against all 3 strategies.
We measure retrieval score for each — higher is better.
This is how engineers pick the best chunking strategy in production.

In [7]:
def build_faiss_index(chunks):
    """
    Given a list of text chunks, embed them and store in FAISS.
    Returns the index and embeddings for searching.
    """
    # Embed all chunks
    embeddings = model.encode(chunks)
    embeddings_float = np.array(embeddings).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(embeddings_float)

    # Build FAISS index
    index = faiss.IndexFlatIP(384)
    index.add(embeddings_float)

    return index

def search_index(index, chunks, question, k=1):
    """
    Search FAISS index with a question.
    Returns the top chunk and its score.
    """
    # Embed the question
    q_embedding = model.encode([question])
    q_float = np.array(q_embedding).astype("float32")
    faiss.normalize_L2(q_float)

    # Search
    distances, indices = index.search(q_float, k=k)

    top_score = distances[0][0]
    top_chunk = chunks[indices[0][0]]

    return top_score, top_chunk

# Build FAISS index for each strategy
print("Building indexes for all 3 strategies...")
index_fixed = build_faiss_index(fixed_chunks)
index_sentence = build_faiss_index(sentence_chunk_list)
index_sliding = build_faiss_index(sliding_chunks)
print("All 3 indexes built!\n")

# Test questions — one per policy section
test_questions = [
    "What documents do I need to become a delivery partner?",
    "How much time do I have to raise a payment dispute?",
    "What should I do if a customer is not reachable?",
    "What happens if I use abusive language with a customer?"
]

# Compare all 3 strategies on each question
print(f"{'Question':<45} {'Fixed':>8} {'Sentence':>10} {'Sliding':>9}")
print("-" * 75)

for question in test_questions:
    score_fixed, _ = search_index(index_fixed, fixed_chunks, question)
    score_sentence, _ = search_index(index_sentence, sentence_chunk_list, question)
    score_sliding, _ = search_index(index_sliding, sliding_chunks, question)

    # Truncate question for display
    q_display = question[:44]

    print(f"{q_display:<45} {score_fixed:>8.4f} {score_sentence:>10.4f} {score_sliding:>9.4f}")

Building indexes for all 3 strategies...
All 3 indexes built!

Question                                         Fixed   Sentence   Sliding
---------------------------------------------------------------------------
What documents do I need to become a deliver    0.5216     0.5641    0.5486
How much time do I have to raise a payment d    0.4251     0.5775    0.5384
What should I do if a customer is not reacha    0.3519     0.6582    0.5623
What happens if I use abusive language with     0.4924     0.6308    0.5009


## Step 7 - Use Winning Strategy to Answer Questions
Sentence chunking gave best retrieval scores.
We now use it to answer all 4 questions through the full RAG pipeline.

In [8]:
import os
from groq import Groq
from google.colab import userdata

# Load Groq
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

def rag_answer(question, index, chunks, top_k=2):
    """
    Full RAG pipeline in one function.
    1. Embed question
    2. Retrieve top_k chunks from FAISS
    3. Send to LLM for grounded answer
    """
    # Embed and normalize question
    q_embedding = model.encode([question])
    q_float = np.array(q_embedding).astype("float32")
    faiss.normalize_L2(q_float)

    # Retrieve top_k chunks
    distances, indices = index.search(q_float, k=top_k)

    # Build context from retrieved chunks
    retrieved = [chunks[idx] for idx in indices[0]]
    context = "\n\n".join(retrieved)
    top_score = distances[0][0]

    # Send to LLM
    prompt = f"""You are a Swiggy delivery partner support agent.

Use the following policy context to answer the question.
Only use the context provided — do not make up information.
If the answer is not in the context, say "I could not find this information."

Policy Context:
{context}

Question: {question}

Answer clearly in 2 sentences."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "question": question,
        "top_score": top_score,
        "answer": response.choices[0].message.content
    }

# Test all 4 questions using sentence chunking strategy
print("RAG Pipeline Results — Sentence Chunking Strategy")
print("=" * 60)

for question in test_questions:
    result = rag_answer(question, index_sentence, sentence_chunk_list)
    print(f"\nQuestion: '{result['question']}'")
    print(f"Retrieval Score: {result['top_score']:.4f}")
    print(f"Answer: {result['answer']}")
    print("-" * 60)

RAG Pipeline Results — Sentence Chunking Strategy

Question: 'What documents do I need to become a delivery partner?'
Retrieval Score: 0.5641
Answer: I could not find this information in the provided policy context. This suggests that Swiggy's delivery partner requirements and documents may not be specified in this context.
------------------------------------------------------------

Question: 'How much time do I have to raise a payment dispute?'
Retrieval Score: 0.5775
Answer: You have 7 days from the date of the disputed payment to raise a payment dispute through the partner support portal. 

All grievances will be acknowledged within 24 hours of submission and this time frame is a general acknowledgement time frame which may not be directly related to payment dispute but it can also be applied accordingly.
------------------------------------------------------------

Question: 'What should I do if a customer is not reachable?'
Retrieval Score: 0.6582
Answer: If a customer is not re

## What We Built
- Implemented 3 chunking strategies: Fixed Size, Sentence, Sliding Window
- Built a reusable FAISS index builder and search function
- Compared retrieval scores across all 3 strategies on 4 test questions
- Identified Sentence Chunking as the winner for our Swiggy policy document
- Built a clean reusable rag_answer() function for the full pipeline
- Learned that chunk quality directly controls retrieval quality

## Chunking Strategy Comparison
| Strategy       | Avg Score | Best For                          |
|----------------|-----------|-----------------------------------|
| Fixed Size     | 0.44      | Never — breaks sentences          |
| Sentence       | 0.60      | Short focused policy documents    |
| Sliding Window | 0.54      | Large docs with context boundaries|

## Key Lessons Learned
- Smaller focused chunks = sharper embeddings = better retrieval
- Fixed size chunking breaks sentences and dilutes meaning
- Sentence chunking gives the sharpest signal for focused questions
- Sliding window helps when context spans multiple sentences
- Knowledge base gaps cause failures — not the algorithm
- DRY principle — build reusable functions, not repeated code

## FAANG Interview Answer
"Chunk size is one of the most important hyperparameters in a RAG pipeline.
Large chunks dilute the embedding signal. Small sentence chunks give sharp,
focused embeddings that retrieve accurately. In production I would test
multiple chunking strategies and measure retrieval scores to pick the best one."

## AWS Equivalent
| What we did here         | AWS service                          |
|--------------------------|--------------------------------------|
| Fixed / Sentence chunks  | Amazon Bedrock Knowledge Bases       |
|                          | (supports fixed and semantic chunking)|
| Sliding window chunking  | Custom Lambda chunking function      |
| FAISS index per strategy | Amazon OpenSearch Serverless         |
| Retrieval score comparison| Amazon Bedrock Model Evaluation     |

## What is Next
Session 5 — RAG Evaluation
We measure our pipeline